In [ ]:
!mkdir -p .github/workflows

In [ ]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python Batch

# ¿Cuándo se dispara el robot? Cuando hacemos un push a la rama main
on:
  push:
    branches: [ main ]

jobs:
  build-and-test:
    runs-on: ubuntu-latest # Usará una máquina Linux en la nube para el ensamble

    steps:
    # Paso 1: Descargar el código del repositorio de GitHub
    - name: Descargar código
      uses: actions/checkout@v4

    # Paso 2: Configurar el entorno de Python en la máquina virtual
    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    # Paso 3: Instalar dependencias del proyecto
    - name: Instalar dependencias
      run: |
        python -m pip install --upgrade pip
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    # Paso 4: Validar la sintaxis del código (Linting)
    - name: Validar sintaxis con Flake8
      run: |
        pip install flake8
        # Detiene la ejecución si hay errores de sintaxis de Python
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

In [ ]:
!ls -la .github/workflows/

In [ ]:
!ls

In [ ]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python Batch (LocalStack Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    # Definimos LocalStack como un servicio que correrá en segundo plano en la máquina de GitHub
    services:
      localstack:
        image: localstack/localstack:latest
        ports:
          - 4566:4566
        env:
          SERVICES: ecr,s3  # Activamos ECR y S3 simulados

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias
      run: |
        python -m pip install --upgrade pip
        pip install awscli awscli-local  # Instalamos herramientas de AWS para interactuar con LocalStack
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        pip install flake8
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Probar conexión y crear repositorio ECR en LocalStack
      run: |
        # Esperamos un momento a que LocalStack termine de levantar en el puerto 4566
        sleep 5
        
        # Configuramos credenciales ficticias necesarias para AWS CLI
        export AWS_ACCESS_KEY_ID=test
        export AWS_SECRET_ACCESS_KEY=test
        export AWS_DEFAULT_REGION=us-east-1
        
        # Creamos el repositorio de imágenes ECR simulado dentro del LocalStack de GitHub
        aws --endpoint-url=http://localhost:4566 ecr create-repository --repository-name mi-proyecto-batch-repo
        
        # Listamos los repositorios para verificar que se creó con éxito en el log de GitHub
        aws --endpoint-url=http://localhost:4566 ecr describe-repositories

    - name: Construir Imagen de Docker
      run: |
        # Simulamos la construcción de la imagen de tu proceso Batch
        # (Asumiendo que tienes un Dockerfile en tu proyecto)
        if [ -f Dockerfile ]; then
          docker build -t mi-proyecto-batch:latest .
        else
          echo "Dockerfile no encontrado, creando uno mínimo temporal para la prueba"
          echo "FROM python:3.11-slim" > Dockerfile
          echo "CMD [\"python\", \"-c\", \"print('Batch Ejecutado con Éxito')\"]" >> Dockerfile
          docker build -t mi-proyecto-batch:latest .
        fi

    - name: Taggear y Empujar la Imagen a LocalStack (ECR Simulado)
      run: |
        # En AWS ECR real, la URL tiene una estructura específica. En LocalStack la emulamos así:
        docker tag mi-proyecto-batch:latest localhost:4566/mi-proyecto-batch-repo:latest
        
        # Empujamos la imagen al puerto simulado de LocalStack en GitHub
        docker push localhost:4566/mi-proyecto-batch-repo:latest
        
        echo "¡Imagen de Docker empujada con éxito al ECR simulado en GitHub Actions!"

In [ ]:
%%writefile requirements.txt
fastapi==0.110.0
uvicorn==0.28.0

In [ ]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python API (LocalStack Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    services:
      localstack:
        image: localstack/localstack:latest
        ports:
          - 4566:4566
        env:
          SERVICES: ecr

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install awscli awscli-local flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga horrores de sintaxis antes de compilar
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Crear repositorio ECR en LocalStack
      run: |
        sleep 5
        export AWS_ACCESS_KEY_ID=test
        export AWS_SECRET_ACCESS_KEY=test
        export AWS_DEFAULT_REGION=us-east-1
        
        # Creamos el almacén para nuestra imagen de FastAPI
        aws --endpoint-url=http://localhost:4566 ecr create-repository --repository-name mi-api-fastapi-repo

    - name: Construir Imagen de Docker de tu API
      run: |
        # Docker usará tu Dockerfile real que copia tu main.py e instala FastAPI/Uvicorn
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen a LocalStack (ECR Simulado)
      run: |
        docker tag mi-api-fastapi:latest localhost:4566/mi-api-fastapi-repo:latest
        docker push localhost:4566/mi-api-fastapi-repo:latest
        echo "¡Imagen de tu API de FastAPI construida y guardada exitosamente en ECR simulado!"

In [ ]:
# 1. Aseguramos agregar los nuevos archivos y el pipeline actualizado
!git add main.py Dockerfile requirements.txt .github/workflows/ci-cd-pipeline.yml

# 2. Creamos un commit descriptivo
!git commit -m "feat: integrar FastAPI, Dockerfile y pipeline simplificado de LocalStack"

# 3. Empujamos los cambios a GitHub
!git push origin main

In [ ]:
!pwd

In [ ]:
!ls -F

In [ ]:
!git status || (cd ../.. && git status)

In [ ]:
# 1. Inicializar el repositorio Git local
!git init

# 2. Asegurarnos de que la rama por defecto se llame main
!git branch -M main

# 3. Agregar todos tus archivos al área de preparación (stage)
!git add main.py Dockerfile requirements.txt .github/workflows/ci-cd-pipeline.yml

# 4. Hacer tu primer commit local
!git commit -m "feat: inicializar proyecto fastapi con Docker y CI/CD"

In [ ]:
!git branch -M main

In [ ]:
# Registrar tu firma
!git config --global user.name "Leonardo Pinzon"
!git config --global user.email "leonardopinzon91@gmail.com"

In [ ]:
# 1. Vincular la carpeta local con la de la nube
!git remote add origin https://github.com/leopinzon75/architect-python-api.git

# 2. Empujar los archivos a GitHub
!git push -u origin main

In [ ]:
!curl -s "https://api.github.com/search/users?q=email:tu_correo_personal@gmail.com" | grep -m 1 '"login":'

In [ ]:
!git remote add origin https://github.com/leopinzon75/architect-python-api.git

In [ ]:
!git push -u origin main

In [ ]:
git push -u origin main

In [ ]:
user1s-MacBook-Pro:ninth_container admin$ git push -u origin main
Counting objects: 8, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (5/5), done.
Writing objects: 100% (8/8), 1.72 KiB | 883.00 KiB/s, done.
Total 8 (delta 0), reused 0 (delta 0)
To https://github.com/leopinzon75/architect-python-api.git
 * [new branch]      main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.
user1s-MacBook-Pro:ninth_container admin$ 

In [ ]:
Password for 'https://leopinzon75@github.com': 
Counting objects: 8, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (5/5), done.
Writing objects: 100% (8/8), 1.72 KiB | 441.00 KiB/s, done.
Total 8 (delta 0), reused 0 (delta 0)
To https://github.com/leopinzon75/architect-python-api.git
 ! [remote rejected] main -> main (refusing to allow a Personal Access Token to create or update workflow `.github/workflows/ci-cd-pipeline.yml` without `workflow` scope)
error: failed to push some refs to 'https://github.com/leopinzon75/architect-python-api.git'
user1s-MacBook-Pro:ninth_container admin$ 

In [ ]:
!cat .github/workflows/ci-cd-pipeline.yml

In [ ]:
name: Pipeline CI/CD - Architect Python API (LocalStack Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install awscli awscli-local flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga errores de sintaxis antes de compilar
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Iniciar LocalStack en segundo plano
      run: |
        # Arrancamos LocalStack directamente con Docker sin bloquear el pipeline
        docker run -d --name localstack -p 4566:4566 -e SERVICES=ecr localstack/localstack:latest
        
        # Esperamos a que LocalStack responda en el puerto 4566 de forma segura
        echo "Esperando que LocalStack esté listo..."
        timeout 30s bash -c 'until curl -s http://localhost:4566/_localstack/health | grep -q "\"ecr\":"; do sleep 2; done' || true
        echo "¡LocalStack inicializado y listo para recibir llamadas!"

    - name: Crear repositorio ECR en LocalStack
      run: |
        export AWS_ACCESS_KEY_ID=test
        export AWS_SECRET_ACCESS_KEY=test
        export AWS_DEFAULT_REGION=us-east-1
        
        # Creamos el almacén para nuestra imagen de FastAPI en el puerto local
        aws --endpoint-url=http://localhost:4566 ecr create-repository --repository-name mi-api-fastapi-repo

    - name: Construir Imagen de Docker de tu API
      run: |
        # Docker usará tu Dockerfile real que copia tu main.py e instala FastAPI/Uvicorn
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen a LocalStack (ECR Simulado)
      run: |
        docker tag mi-api-fastapi:latest localhost:4566/mi-api-fastapi-repo:latest
        docker push localhost:4566/mi-api-fastapi-repo:latest
        echo "¡Imagen de tu API de FastAPI construida y guardada exitosamente en ECR simulado!"

In [ ]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python API (LocalStack Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install awscli awscli-local flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga errores de sintaxis antes de compilar
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Iniciar LocalStack en segundo plano
      run: |
        # Arrancamos LocalStack directamente con Docker sin bloquear el pipeline
        docker run -d --name localstack -p 4566:4566 -e SERVICES=ecr localstack/localstack:latest
        
        # Esperamos a que LocalStack responda en el puerto 4566 de forma segura
        echo "Esperando que LocalStack esté listo..."
        timeout 30s bash -c 'until curl -s http://localhost:4566/_localstack/health | grep -q "\"ecr\":"; do sleep 2; done' || true
        echo "¡LocalStack inicializado y listo para recibir llamadas!"

    - name: Crear repositorio ECR en LocalStack
      run: |
        export AWS_ACCESS_KEY_ID=test
        export AWS_SECRET_ACCESS_KEY=test
        export AWS_DEFAULT_REGION=us-east-1
        
        # Creamos el almacén para nuestra imagen de FastAPI en el puerto local
        aws --endpoint-url=http://localhost:4566 ecr create-repository --repository-name mi-api-fastapi-repo

    - name: Construir Imagen de Docker de tu API
      run: |
        # Docker usará tu Dockerfile real que copia tu main.py e instala FastAPI/Uvicorn
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen a LocalStack (ECR Simulado)
      run: |
        docker tag mi-api-fastapi:latest localhost:4566/mi-api-fastapi-repo:latest
        docker push localhost:4566/mi-api-fastapi-repo:latest
        echo "¡Imagen de tu API de FastAPI construida y guardada exitosamente en ECR simulado!"

In [ ]:
git add .github/workflows/ci-cd-pipeline.yml
git commit -m "fix: optimizar arranque de localstack en el pipeline"
git push origin main

In [ ]:
git add .github/workflows/ci-cd-pipeline.yml
git commit -m "fix: mount docker socket and add debugging logs for localstack"
git push origin main

In [ ]:
output: 

To https://github.com/leopinzon75/architect-python-api.git
 * [new branch]      main -> main
Branch 'main' set up to track remote branch 'main' from 'origin'.
user1s-MacBook-Pro:ninth_container admin$ git add .github/workflows/ci-cd-pipeline.yml
user1s-MacBook-Pro:ninth_container admin$ git commit -m "fix: optimizar arranque de localstack en el pipeline"
[main 62ff72a] fix: optimizar arranque de localstack en el pipeline
 1 file changed, 12 insertions(+), 11 deletions(-)
user1s-MacBook-Pro:ninth_container admin$ git add .github/workflows/ci-cd-pipeline.yml
user1s-MacBook-Pro:ninth_container admin$ git commit -m "fix: optimizar arranque de localstack en el pipeline"
On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Untracked files:
	.ipynb_checkpoints/
	"fase_7_automatizaci\303\263n_total_con_CI_CD.ipynb"

nothing added to commit but untracked files present
user1s-MacBook-Pro:ninth_container admin$ git push origin main

In [ ]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python API (LocalStack Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install awscli awscli-local flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga errores de sintaxis antes de compilar
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Iniciar LocalStack en segundo plano
      run: |
        # Arrancamos LocalStack en segundo plano
        docker run -d --name localstack -p 4566:4566 -e SERVICES=ecr localstack/localstack:latest
        
        # Esperamos de forma segura hasta que el puerto 4566 responda activamente
        echo "Esperando que el puerto de LocalStack (4566) esté listo..."
        for i in {1..15}; do
          if curl -s http://localhost:4566/health | grep -q "\"ecr\":"; then
            echo "¡LocalStack y ECR están completamente listos!"
            exit 0
          fi
          echo "LocalStack aún inicializando... esperando 3 segundos más (Intento $i de 15)"
          sleep 3
        done
        echo "Advertencia: Se alcanzó el límite de espera, continuando de todos modos."

    - name: Crear repositorio ECR en LocalStack
      run: |
        export AWS_ACCESS_KEY_ID=test
        export AWS_SECRET_ACCESS_KEY=test
        export AWS_DEFAULT_REGION=us-east-1
        
        # Creamos el almacén para nuestra imagen de FastAPI en el puerto local
        aws --endpoint-url=http://localhost:4566 ecr create-repository --repository-name mi-api-fastapi-repo

    - name: Construir Imagen de Docker de tu API
      run: |
        # Docker usará tu Dockerfile real que copia tu main.py e instala FastAPI/Uvicorn
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen a LocalStack (ECR Simulado)
      run: |
        docker tag mi-api-fastapi:latest localhost:4566/mi-api-fastapi-repo:latest
        docker push localhost:4566/mi-api-fastapi-repo:latest
        echo "¡Imagen de tu API de FastAPI construida y guardada exitosamente en ECR simulado!"

In [ ]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python API (LocalStack Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install awscli awscli-local flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga errores de sintaxis antes de compilar
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Iniciar LocalStack en segundo plano
      run: |
        # Arrancamos LocalStack mapeando el puerto de manera global en el runner
        docker run -d --name localstack -p 4566:4566 -e SERVICES=ecr localstack/localstack:latest
        
        # Espera fija inicial para permitir que el motor levante los sockets
        echo "Esperando inicialización de los servicios de LocalStack..."
        sleep 20
        
        # Verificación directa de disponibilidad del puerto
        docker ps -a
        echo "¡LocalStack listo para recibir peticiones!"

    - name: Crear repositorio ECR en LocalStack
      run: |
        export AWS_ACCESS_KEY_ID=test
        export AWS_SECRET_ACCESS_KEY=test
        export AWS_DEFAULT_REGION=us-east-1
        
        # Creamos el almacén para nuestra imagen de FastAPI en el puerto local
        aws --endpoint-url=http://127.0.0.1:4566 ecr create-repository --repository-name mi-api-fastapi-repo

    - name: Construir Imagen de Docker de tu API
      run: |
        # Docker usará tu Dockerfile real que copia tu main.py e instala FastAPI/Uvicorn
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen a LocalStack (ECR Simulado)
      run: |
        docker tag mi-api-fastapi:latest 127.0.0.1:4566/mi-api-fastapi-repo:latest
        docker push 127.0.0.1:4566/mi-api-fastapi-repo:latest
        echo "¡Imagen de tu API de FastAPI construida y guardada exitosamente en ECR simulado!"

In [ ]:
Run # Arrancamos LocalStack mapeando el puerto de manera global en el runner
Unable to find image 'localstack/localstack:latest' locally
latest: Pulling from localstack/localstack
0b83a2e2494a: Pulling fs layer
0132b37780a2: Pulling fs layer
12d752851a01: Pulling fs layer
742a6a10bc4d: Pulling fs layer
4f4fb700ef54: Pulling fs layer
189ec99bcb00: Pulling fs layer
b4a96f9ced05: Pulling fs layer
c8bcd4f18d31: Pulling fs layer
b41453157897: Pulling fs layer
fc6f576d8f09: Pulling fs layer
825cdc201eba: Pulling fs layer
c678ccd99994: Pulling fs layer
bb5461fdb523: Pulling fs layer
a4b822c8e3ce: Pulling fs layer
d003fdee37dd: Pulling fs layer
61ae72db49b1: Pulling fs layer
d5c50a919b60: Pulling fs layer
53cdd092db92: Pulling fs layer
1faa630a726b: Pulling fs layer
30bfc3ad9a11: Pulling fs layer
b41453157897: Waiting
fc6f576d8f09: Waiting
825cdc201eba: Waiting
c678ccd99994: Waiting
bb5461fdb523: Waiting
a4b822c8e3ce: Waiting
53cdd092db92: Waiting
1faa630a726b: Waiting
30bfc3ad9a11: Waiting
d003fdee37dd: Waiting
61ae72db49b1: Waiting
742a6a10bc4d: Waiting
189ec99bcb00: Waiting
4f4fb700ef54: Waiting
b4a96f9ced05: Waiting
d5c50a919b60: Waiting
c8bcd4f18d31: Waiting
0b83a2e2494a: Verifying Checksum
0b83a2e2494a: Download complete
742a6a10bc4d: Verifying Checksum
742a6a10bc4d: Download complete
0132b37780a2: Verifying Checksum
0132b37780a2: Download complete
12d752851a01: Verifying Checksum
12d752851a01: Download complete
189ec99bcb00: Verifying Checksum
189ec99bcb00: Download complete
4f4fb700ef54: Verifying Checksum
4f4fb700ef54: Download complete
c8bcd4f18d31: Verifying Checksum
c8bcd4f18d31: Download complete
b41453157897: Verifying Checksum
b41453157897: Download complete
b4a96f9ced05: Verifying Checksum
b4a96f9ced05: Download complete
0b83a2e2494a: Pull complete
c678ccd99994: Verifying Checksum
c678ccd99994: Download complete
fc6f576d8f09: Verifying Checksum
fc6f576d8f09: Download complete
bb5461fdb523: Verifying Checksum
bb5461fdb523: Download complete
a4b822c8e3ce: Verifying Checksum
a4b822c8e3ce: Download complete
825cdc201eba: Download complete
d003fdee37dd: Verifying Checksum
d003fdee37dd: Download complete
d5c50a919b60: Verifying Checksum
d5c50a919b60: Download complete
53cdd092db92: Verifying Checksum
53cdd092db92: Download complete
61ae72db49b1: Verifying Checksum
61ae72db49b1: Download complete
30bfc3ad9a11: Download complete
1faa630a726b: Verifying Checksum
1faa630a726b: Download complete
0132b37780a2: Pull complete
12d752851a01: Pull complete
742a6a10bc4d: Pull complete
4f4fb700ef54: Pull complete
189ec99bcb00: Pull complete
b4a96f9ced05: Pull complete
c8bcd4f18d31: Pull complete
b41453157897: Pull complete
fc6f576d8f09: Pull complete
825cdc201eba: Pull complete
c678ccd99994: Pull complete
bb5461fdb523: Pull complete
a4b822c8e3ce: Pull complete
d003fdee37dd: Pull complete
61ae72db49b1: Pull complete
d5c50a919b60: Pull complete
53cdd092db92: Pull complete
1faa630a726b: Pull complete
30bfc3ad9a11: Pull complete
Digest: sha256:6f0e0b9486b5b3621effad94b0149e5683a83dcd7f274c25b283ae69d3cc81ed
Status: Downloaded newer image for localstack/localstack:latest
305932fceceea36c856f148993924f3e1daecff9b7d79b1ff9348b0a0f344479
Esperando inicialización de los servicios de LocalStack...
CONTAINER ID   IMAGE                          COMMAND                  CREATED          STATUS                       PORTS     NAMES
305932fcecee   localstack/localstack:latest   "docker-entrypoint.sh"   20 seconds ago   Exited (55) 19 seconds ago             localstack
¡LocalStack listo para recibir peticiones!

In [ ]:
- name: Start LocalStack
        run: |
          # Arrancamos LocalStack con lo mínimo indispensable para que no falle
          docker run -d \
            --name localstack \
            -p 4566:4566 \
            -e SERVICES=ecr,s3 \
            -e DEBUG=1 \
            localstack/localstack:latest

          echo "Esperando a que LocalStack intente iniciar..."
          sleep 10

          # Verificamos si sigue vivo o si falló, imprimiendo sus logs
          echo "=== ESTADO DEL CONTENEDOR ==="
          docker ps -a --filter name=localstack
          
          echo "=== LOGS DE LOCALSTACK ==="
          docker logs localstack

          echo "Esperando 10 segundos adicionales para completar la inicialización..."
          sleep 10

In [ ]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python API (LocalStack Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install awscli awscli-local flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga errores de sintaxis antes de compilar
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Iniciar LocalStack en segundo plano
      run: |
        # Arrancamos LocalStack con el socket de Docker montado y DEBUG activo
        docker run -d \
          --name localstack \
          -p 4566:4566 \
          -v /var/run/docker.sock:/var/run/docker.sock \
          -e SERVICES=ecr \
          -e DEBUG=1 \
          localstack/localstack:latest
        
        # Espera inicial para que el contenedor intente levantar
        echo "Esperando 15 segundos para la inicialización inicial..."
        sleep 15
        
        # Inspección preventiva de estado y logs
        echo "=== ESTADO DEL CONTENEDOR ==="
        docker ps -a
        
        echo "=== LOGS DE LOCALSTACK ==="
        docker logs localstack
        
        echo "Esperando 5 segundos adicionales para asegurar puertos..."
        sleep 5
        echo "¡LocalStack listo para recibir peticiones!"

    - name: Crear repositorio ECR en LocalStack
      run: |
        export AWS_ACCESS_KEY_ID=test
        export AWS_SECRET_ACCESS_KEY=test
        export AWS_DEFAULT_REGION=us-east-1
        
        # Creamos el almacén para nuestra imagen de FastAPI en el puerto local
        aws --endpoint-url=http://127.0.0.1:4566 ecr create-repository --repository-name mi-api-fastapi-repo

    - name: Construir Imagen de Docker de tu API
      run: |
        # Docker usará tu Dockerfile real que copia tu main.py e instala FastAPI/Uvicorn
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen a LocalStack (ECR Simulado)
      run: |
        docker tag mi-api-fastapi:latest 127.0.0.1:4566/mi-api-fastapi-repo:latest
        docker push 127.0.0.1:4566/mi-api-fastapi-repo:latest
        echo "¡Imagen de tu API de FastAPI construida y guardada exitosamente en ECR simulado!"

In [ ]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python API (LocalStack Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install awscli awscli-local flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga errores de sintaxis antes de compilar
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Iniciar LocalStack en segundo plano
      run: |
        # Usamos la versión 3.4.0 (estable y libre de problemas de licencia Pro)
        # y desactivamos explícitamente la activación Pro.
        docker run -d \
          --name localstack \
          -p 4566:4566 \
          -v /var/run/docker.sock:/var/run/docker.sock \
          -e SERVICES=ecr \
          -e ACTIVATE_PRO=0 \
          -e DEBUG=1 \
          localstack/localstack:3.4.0
        
        # Espera inicial para que el contenedor intente levantar
        echo "Esperando 15 segundos para la inicialización inicial..."
        sleep 15
        
        # Inspección preventiva de estado y logs
        echo "=== ESTADO DEL CONTENEDOR ==="
        docker ps -a
        
        echo "=== LOGS DE LOCALSTACK ==="
        docker logs localstack
        
        echo "Esperando 5 segundos adicionales para asegurar puertos..."
        sleep 5
        echo "¡LocalStack listo para recibir peticiones!"

    - name: Crear repositorio ECR en LocalStack
      run: |
        export AWS_ACCESS_KEY_ID=test
        export AWS_SECRET_ACCESS_KEY=test
        export AWS_DEFAULT_REGION=us-east-1
        
        # Creamos el almacén para nuestra imagen de FastAPI en el puerto local
        aws --endpoint-url=http://127.0.0.1:4566 ecr create-repository --repository-name mi-api-fastapi-repo

    - name: Construir Imagen de Docker de tu API
      run: |
        # Docker usará tu Dockerfile real que copia tu main.py e instala FastAPI/Uvicorn
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen a LocalStack (ECR Simulado)
      run: |
        docker tag mi-api-fastapi:latest 127.0.0.1:4566/mi-api-fastapi-repo:latest
        docker push 127.0.0.1:4566/mi-api-fastapi-repo:latest
        echo "¡Imagen de tu API de FastAPI construida y guardada exitosamente en ECR simulado!"

In [ ]:
git add .github/workflows/ci-cd-pipeline.yml
git commit -m "fix: mount docker socket and add debugging logs for localstack"
git push origin main

In [ ]:
git add .
git commit -m "fix: disable localstack pro activation and pin stable version 3.4.0"
git push origin main

In [18]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python API (LocalStack Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install awscli awscli-local flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga errores de sintaxis antes de compilar
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Iniciar LocalStack en segundo plano
      run: |
        # Usamos la versión 2.0.0, la última con soporte gratuito (Community) para ECR.
        docker run -d \
          --name localstack \
          -p 4566:4566 \
          -v /var/run/docker.sock:/var/run/docker.sock \
          -e SERVICES=ecr \
          localstack/localstack:2.0.0
        
        # Espera para que el contenedor levante
        echo "Esperando 15 segundos para la inicialización..."
        sleep 15
        
        # Inspección preventiva de estado
        echo "=== ESTADO DEL CONTENEDOR ==="
        docker ps -a
        
        echo "¡LocalStack listo para recibir peticiones!"

    - name: Crear repositorio ECR en LocalStack
      run: |
        export AWS_ACCESS_KEY_ID=test
        export AWS_SECRET_ACCESS_KEY=test
        export AWS_DEFAULT_REGION=us-east-1
        
        # Creamos el almacén para nuestra imagen de FastAPI en el puerto local
        aws --endpoint-url=http://127.0.0.1:4566 ecr create-repository --repository-name mi-api-fastapi-repo

    - name: Construir Imagen de Docker de tu API
      run: |
        # Docker usará tu Dockerfile real que copia tu main.py e instala FastAPI/Uvicorn
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen a LocalStack (ECR Simulado)
      run: |
        docker tag mi-api-fastapi:latest 127.0.0.1:4566/mi-api-fastapi-repo:latest
        docker push 127.0.0.1:4566/mi-api-fastapi-repo:latest
        echo "¡Imagen de tu API de FastAPI construida y guardada exitosamente en ECR simulado!"

Overwriting .github/workflows/ci-cd-pipeline.yml


In [ ]:
!git reset --soft HEAD~1

In [1]:
git add fase_7_automatización_total_con_CI_CD.ipynb
git commit -m "chore: remove exposed token and target localstack 2.0.0 for free ecr"

SyntaxError: invalid syntax (4055588062.py, line 1)

In [3]:
!git reset HEAD fase_7_automatización_total_con_CI_CD.ipynb

Unstaged changes after reset:
M	fase_7_automatización_total_con_CI_CD.ipynb


In [5]:
!git add .github/workflows/ci-cd-pipeline.yml

In [6]:
!git commit -m "feat: use localstack 2.0.0 for free ecr"

On branch main
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)

Changes not staged for commit:
	modified:   "fase_7_automatizaci\303\263n_total_con_CI_CD.ipynb"

no changes added to commit


In [7]:
!git push origin main

Counting objects: 7, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (5/5), done.
Writing objects: 100% (7/7), 19.90 KiB | 3.98 MiB/s, done.
Total 7 (delta 1), reused 0 (delta 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
remote: error: GH013: Repository rule violations found for refs/heads/main.
remote: 
remote: - GITHUB PUSH PROTECTION
remote:   —————————————————————————————————————————
remote:     Resolve the following violations before pushing again
remote: 
remote:     - Push cannot contain secrets
remote: 
remote:     
remote:      (?) Learn how to resolve a blocked push
remote:      https://docs.github.com/code-security/secret-scanning/working-with-secret-scanning-and-push-protection/working-with-push-protection-from-the-command-line#resolving-a-blocked-push
remote:     
remote:     
remote:       —— GitHub Personal Access Token ——————————————————————
remote:        locations:
remote:          - commit: b3dd28ecac6bcd301d9fba657

In [8]:
!git stash push .github/workflows/ci-cd-pipeline.yml

No local changes to save


In [9]:
!git reset --hard origin/main

HEAD is now at 40526ad fix: mount docker socket and add debugging logs for localstack


In [11]:
!%%writefile .github/workflows/ci-cd-pipeline.yml

/bin/bash: line 0: fg: no job control


In [12]:
!git add .github/workflows/ci-cd-pipeline.yml

In [14]:
!git commit -m "feat: use localstack 2.0.0 for free ecr support"

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
	.ipynb_checkpoints/
	"fase_7_automatizaci\303\263n_total_con_CI_CD.ipynb"

nothing added to commit but untracked files present


In [15]:
!git push origin main

Everything up-to-date


In [16]:
!git add .github/workflows/ci-cd-pipeline.yml
!git commit -m "feat: update localstack to 2.0.0 for free ecr"
!git push origin main

On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
	.ipynb_checkpoints/
	"fase_7_automatizaci\303\263n_total_con_CI_CD.ipynb"

nothing added to commit but untracked files present
Everything up-to-date


In [17]:
nano .github/workflows/ci-cd-pipeline.yml

^X Exit^J Justify   ^W Where Is  ^V Next Page ^U UnCut Text^T To Spell5;7Huses: actions/checkout@v4- name: Configurar Pythonuses: actions/setup-python@v5with:python-version: '3.11' [ Read 77 lines ]Modified[ Can now UnJustify! ]Justify 

In [ ]:
@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

In [19]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python API (Docker Registry Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga errores de sintaxis
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Iniciar Registro Local de Docker
      run: |
        # Levantamos un registro Docker oficial y gratuito en el puerto 5000
        docker run -d -p 5000:5000 --name registro-local registry:2
        
        echo "Esperando 5 segundos para inicializar el registro..."
        sleep 5

    - name: Construir Imagen de Docker de tu API
      run: |
        # Construimos la imagen usando tu Dockerfile real
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen al Registro Local
      run: |
        # Taggeamos la imagen apuntando a nuestro registro local en el puerto 5000
        docker tag mi-api-fastapi:latest localhost:5000/mi-api-fastapi-repo:latest
        
        # Subimos la imagen para simular el "push" de AWS ECR
        docker push localhost:5000/mi-api-fastapi-repo:latest
        echo "¡Imagen de FastAPI subida con éxito al registro simulado!"

Overwriting .github/workflows/ci-cd-pipeline.yml


In [20]:
%%writefile .github/workflows/ci-cd-pipeline.yml
name: Pipeline CI/CD - Architect Python API (Docker Registry Test)

on:
  push:
    branches: [ main ]

jobs:
  build-test-and-local-deploy:
    runs-on: ubuntu-latest

    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Instalar dependencias locales de prueba
      run: |
        python -m pip install --upgrade pip
        pip install flake8
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Validar sintaxis con Flake8
      run: |
        # Validamos que tu main.py no tenga errores de sintaxis
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

    - name: Iniciar Registro Local de Docker
      run: |
        # Levantamos un registro Docker oficial y gratuito en el puerto 5000
        docker run -d -p 5000:5000 --name registro-local registry:2
        
        echo "Esperando 5 segundos para inicializar el registro..."
        sleep 5

    - name: Construir Imagen de Docker de tu API
      run: |
        # Construimos la imagen usando tu Dockerfile real
        docker build -t mi-api-fastapi:latest .

    - name: Taggear y Empujar la Imagen al Registro Local
      run: |
        # Taggeamos la imagen apuntando a nuestro registro local en el puerto 5000
        docker tag mi-api-fastapi:latest localhost:5000/mi-api-fastapi-repo:latest
        
        # Subimos la imagen para simular el "push" de AWS ECR
        docker push localhost:5000/mi-api-fastapi-repo:latest
        echo "¡Imagen de FastAPI subida con éxito al registro simulado!"

Overwriting .github/workflows/ci-cd-pipeline.yml


In [ ]:
##################################
Ahora CD despliegue continuo:


In [1]:
import os
print(os.getcwd())

/Users/admin/Desktop/Shafer_Python_Classes/Cloud_engineering_practice/containers/ninth_container


In [2]:
!ls

Dockerfile
ejercicio_2
fase_7_automatización_total_con_CI_CD.ipynb
main.py
requirements.txt


In [ ]:
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"Hello": "Youtube"}


In [3]:
%%writefile requirements.txt
fastapi==0.110.0
uvicorn==0.28.0

Overwriting requirements.txt


In [4]:
%%writefile ../.github/workflows/ci-cd-ejercicio1.yml
name: Pipeline CI/CD - Fleet Auth API (Ejercicio 1)

on:
  push:
    branches: [ main ]
    # Este filtro asegura que el pipeline solo corra si cambias cosas de este ejercicio
    paths:
      - 'main.py'
      - 'Dockerfile'
      - 'requirements.txt'

jobs:
  build-and-publish:
    runs-on: ubuntu-latest
    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Iniciar Sesión de forma Segura en Docker Hub
      uses: docker/login-action@v3
      with:
        username: ${{ secrets.DOCKER_USERNAME }}
        password: ${{ secrets.DOCKER_PASSWORD }}

    - name: Construir y Empujar Imagen de la API a Docker Hub
      run: |
        docker build -t ${{ secrets.DOCKER_USERNAME }}/fleet-auth-api:latest .
        docker push ${{ secrets.DOCKER_USERNAME }}/fleet-auth-api:latest
        echo "¡Imagen de la API de Autenticación empujada exitosamente!"

Writing ../.github/workflows/ci-cd-ejercicio1.yml


FileNotFoundError: [Errno 2] No such file or directory: '../.github/workflows/ci-cd-ejercicio1.yml'

In [5]:
!git add requirements.txt ../.github/workflows/ci-cd-ejercicio1.yml
!git commit -m "feat: fix requirements.txt and add CI/CD pipeline for Ejercicio 1"
!git push origin main

fatal: ../.github/workflows/ci-cd-ejercicio1.yml: '../.github/workflows/ci-cd-ejercicio1.yml' is outside repository
On branch main
Your branch is up to date with 'origin/main'.

Untracked files:
	.ipynb_checkpoints/
	ejercicio_2/.ipynb_checkpoints/
	ejercicio_2/data/
	ejercicio_2/ejer_2.ipynb
	"fase_7_automatizaci\303\263n_total_con_CI_CD.ipynb"

nothing added to commit but untracked files present
Everything up-to-date


In [6]:
!ls -la ..

total 24
drwxr-xr-x  14 admin  staff   448 Jul 11 23:17 .
drwxr-xr-x  13 admin  staff   416 Jul 13 10:58 ..
-rw-r--r--@  1 admin  staff  8196 Jun 28 22:02 .DS_Store
drwxr-xr-x   2 admin  staff    64 Jun 18 14:24 .ipynb_checkpoints
drwxr-xr-x  13 admin  staff   416 Jul  2 16:10 back _up_seventh_container
drwxr-xr-x   8 admin  staff   256 Jul 10 17:42 eighth_container
drwxr-xr-x   6 admin  staff   192 Jun 21 11:53 fifth_container
drwxr-xr-x   9 admin  staff   288 Jul  5 22:04 first_container
drwxr-xr-x   7 admin  staff   224 Jun 20 14:46 fourth_container
drwxr-xr-x  10 admin  staff   320 Jul 15 10:47 ninth_container
drwxr-xr-x   8 admin  staff   256 Jun 18 05:31 second_container
drwxr-xr-x  17 admin  staff   544 Jul  3 18:05 seventh_container
drwxr-xr-x   9 admin  staff   288 Jul  6 11:41 sixth_container
drwxr-xr-x  10 admin  staff   320 Jun 26 10:10 third_container


In [7]:
!ls -la

total 280
drwxr-xr-x  10 admin  staff     320 Jul 15 10:47 .
drwxr-xr-x  14 admin  staff     448 Jul 11 23:17 ..
drwxr-xr-x  13 admin  staff     416 Jul 15 10:46 .git
drwxr-xr-x   3 admin  staff      96 Jul 13 21:26 .github
drwxr-xr-x   6 admin  staff     192 Jul 15 10:37 .ipynb_checkpoints
-rw-r--r--   1 admin  staff     454 Jul 14 15:16 Dockerfile
drwxr-xr-x  10 admin  staff     320 Jul 14 21:58 ejercicio_2
-rw-r--r--   1 admin  staff  127966 Jul 15 10:47 fase_7_automatización_total_con_CI_CD.ipynb
-rw-r--r--   1 admin  staff     109 Jul 14 15:16 main.py
-rw-r--r--   1 admin  staff      33 Jul 15 10:38 requirements.txt


In [8]:
%%writefile .github/workflows/ci-cd-ejercicio1.yml
name: Pipeline CI/CD - Fleet Auth API (Ejercicio 1)

on:
  push:
    branches: [ main ]
    # El pipeline solo correrá si cambias la API raíz
    paths:
      - 'main.py'
      - 'Dockerfile'
      - 'requirements.txt'

jobs:
  build-and-publish:
    runs-on: ubuntu-latest
    steps:
    - name: Descargar código
      uses: actions/checkout@v4

    - name: Configurar Python
      uses: actions/setup-python@v5
      with:
        python-version: '3.11'

    - name: Iniciar Sesión de forma Segura en Docker Hub
      uses: docker/login-action@v3
      with:
        username: ${{ secrets.DOCKER_USERNAME }}
        password: ${{ secrets.DOCKER_PASSWORD }}

    - name: Construir y Empujar Imagen de la API a Docker Hub
      run: |
        docker build -t ${{ secrets.DOCKER_USERNAME }}/fleet-auth-api:latest .
        docker push ${{ secrets.DOCKER_USERNAME }}/fleet-auth-api:latest
        echo "¡Imagen de la API de Autenticación empujada exitosamente!"

Writing .github/workflows/ci-cd-ejercicio1.yml


In [9]:
!git add requirements.txt .github/workflows/ci-cd-ejercicio1.yml
!git commit -m "feat: fix requirements path and add CI/CD pipeline for Ejercicio 1"
!git push origin main

[main ee9c49a] feat: fix requirements path and add CI/CD pipeline for Ejercicio 1
 1 file changed, 34 insertions(+)
 create mode 100644 .github/workflows/ci-cd-ejercicio1.yml
Counting objects: 5, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (4/4), done.
Writing objects: 100% (5/5), 951 bytes | 951.00 KiB/s, done.
Total 5 (delta 1), reused 0 (delta 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/leopinzon75/architect-python-api.git
   83a93fb..ee9c49a  main -> main


In [10]:
%%writefile main.py
from fastapi import FastAPI

app = FastAPI()

@app.get("/")
def read_root():
    return {"Hello": "Youtube", "status": "Pipeline de Ejercicio 1 funcionando perfectamente"}

Overwriting main.py


In [12]:
!git add main.py
!git commit -m "fix: trigger deployment for fleet-auth-api"
!git push origin main

[main 68baaa4] fix: trigger deployment for fleet-auth-api
 1 file changed, 1 insertion(+), 1 deletion(-)
Counting objects: 3, done.
Delta compression using up to 4 threads.
Compressing objects: 100% (3/3), done.
Writing objects: 100% (3/3), 423 bytes | 423.00 KiB/s, done.
Total 3 (delta 1), reused 0 (delta 0)
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/leopinzon75/architect-python-api.git
   ee9c49a..68baaa4  main -> main
